# PretrainTransformer.ipynb

## Giới thiệu
`PretrainTransformer.ipynb` là xây dựng một encoder đơn giản trong transformer phục vụ cho bài toán next token prediction.

## Mô hình
- Sử dụng **word-level tokenization**
- Khởi tạo embedding từ **Word2Vec pretrained (300d)**
- Kiến trúc **Decoder-Only Transformer**
- Gồm:
  - **Token Embedding**
  - **Positional Embedding**
  - **Masked Multi-Head Self-Attention**
  - **Feed Forward Network**
  - **Linear LM Head**
- Huấn luyện theo bài toán **next token prediction**
- Sử dụng **causal mask** để đảm bảo mỗi token chỉ nhìn thấy các token trước đó

## Mục tiêu
Giúp hiểu pipeline của một **LLM mini** từ:

**dữ liệu → tokenization → embedding → masked self-attention → huấn luyện → sinh văn bản**


## Dataset
- Sử dụng **Tiny Shakespeare** làm tập dữ liệu huấn luyện
- Đây là một corpus văn bản nhỏ, thường được dùng để demo các mô hình language modeling và text generation

## Data Preprocessing
- Đọc dữ liệu văn bản từ file `input.txt`
- Tách văn bản thành từng dòng bằng `splitlines()` và loại bỏ các dòng rỗng
- Áp dụng **word-level tokenization** bằng `split()`
- Khởi tạo tập **special tokens** gồm:
  - `<pad>`
  - `<unk>`
  - `<eos>`
- Xây dựng **vocabulary** bằng cách lấy toàn bộ từ duy nhất trong dữ liệu và sắp xếp lại
- Tạo mapping:
  - `stoi`: ánh xạ từ sang chỉ số
  - `itos`: ánh xạ chỉ số sang từ
- Mã hóa dữ liệu văn bản thành tensor số nguyên bằng PyTorch để phục vụ huấn luyện

## Training Configuration
- **Model:** Decoder-Only Transformer
- **Embedding dimension:** `300`
- **Number of heads:** `6`
- **Number of layers:** `4`
- **Dropout:** `0.1`
- **Optimizer:** AdamW
- **Learning rate:** `3 × 10^-4`
- **Training steps:** `3000`
- **Objective:** Next token prediction
- **Loss:** Cross entropy loss
- **Device:** CUDA


In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-13 05:11:54--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-04-13 05:11:54 (36.7 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 38.0 MB/s eta 0:00:00


In [ ]:
import re
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import gensim.downloader as api

In [ ]:
# =========================
# 1. LOAD DATA
# =========================
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read().strip()


In [ ]:
# =========================
# 2. SIMPLE WORD TOKENIZATION
# =========================
# Demo đơn giản:
# - tách câu bằng dấu . ! ? hoặc xuống dòng

special_tokens = ["<pad>", "<unk>", "<eos>"]

lines = [line.strip() for line in text.splitlines() if line.strip()]

all_words = []
for line in lines:
    words = line.split()

    # speaker tag như ROMEO:, JULIET:
    if len(words) == 1 and words[0].endswith(":"):
        all_words.extend(words)
    else:
        all_words.extend(words)

vocab = special_tokens + sorted(set(all_words))
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(vocab)

PAD_ID = stoi["<pad>"]
UNK_ID = stoi["<unk>"]
EOS_ID = stoi["<eos>"]

def encode_words(words):
    return [stoi.get(w, UNK_ID) for w in words]

def encode_text_to_ids(text):
    return encode_words(text.split())

def decode(ids):
    words = []
    for i in ids:
        w = itos[int(i)]
        if w == "<eos>":
            break
        if w != "<pad>":
            words.append(w)
    return " ".join(words)

data = torch.tensor(encode_words(all_words), dtype=torch.long)

In [ ]:
# =========================
# 3. TRAIN/VAL SPLIT
# =========================
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
# =========================
# 4. BATCH FUNCTION
# =========================
block_size = 32
batch_size = 32

def get_batch(split="train"):
    source = train_data if split == "train" else val_data
    ix = torch.randint(0, len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([source[i+1:i+block_size+1] for i in ix])
    return x, y


In [ ]:
# =========================
# 5. LOAD WORD2VEC
# =========================
use_pretrained_w2v = True
embedding_dim = 300

if use_pretrained_w2v:
    print("Loading Word2Vec...")
    w2v = api.load("word2vec-google-news-300")

    embedding_matrix = torch.randn(vocab_size, embedding_dim)

    for word, i in stoi.items():
        if word in special_tokens:
            continue
        if word in w2v:
            embedding_matrix[i] = torch.tensor(w2v[word], dtype=torch.float)
else:
    embedding_matrix = None

Loading Word2Vec...
[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
print(data)

tensor([ 1583,   995,   640,  ..., 22866,  5351, 24458])


In [ ]:
# =========================
# 6. DECODER-ONLY TRANSFORMER
# =========================
n_embd = 300
n_head = 6
n_layer = 4
dropout = 0.1

assert n_embd % n_head == 0

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class DecoderTransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        n_embd=300,
        n_head=6,
        n_layer=4,
        dropout=0.1,
        embedding_matrix=None,
        pad_id=0
    ):
        super().__init__()
        self.block_size = block_size
        self.pad_id = pad_id
        self.n_embd = n_embd

        if embedding_matrix is not None:
            self.token_embedding = nn.Embedding.from_pretrained(
                embedding_matrix,
                freeze=False
            )
        else:
            self.token_embedding = nn.Embedding(vocab_size, n_embd)

        self.position_embedding = nn.Embedding(block_size, n_embd)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=n_head,
            dim_feedforward=4 * n_embd,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layer
        )

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def _generate_causal_mask(self, T, device):
        # shape (T, T), phía trên đường chéo là -inf
        mask = torch.full((T, T), float("-inf"), device=device)
        mask = torch.triu(mask, diagonal=1)
        return mask

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} > block_size {self.block_size}")

        tok_emb = self.token_embedding(idx)  # (B, T, C)
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.position_embedding(pos).unsqueeze(0)  # (1, T, C)

        x = tok_emb + pos_emb

        causal_mask = self._generate_causal_mask(T, idx.device)

        x = self.transformer(
            x,
            mask=causal_mask
        )

        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, C),
                targets.reshape(B * T),
                ignore_index=self.pad_id
            )

        return logits, loss

In [ ]:
# =========================
# 7. TRAIN
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DecoderTransformerLM(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    dropout=dropout,
    embedding_matrix=embedding_matrix
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
max_steps = 3000

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(20):
            xb, yb = get_batch(split)
            xb, yb = xb.to(device), yb.to(device)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

for step in range(max_steps):
    xb, yb = get_batch("train")
    xb, yb = xb.to(device), yb.to(device)

    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        losses = estimate_loss()
        print(
            f"step {step:4d} | "
            f"train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f}"
        )

/tmp/ipykernel_2707/3251342931.py:53: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


step    0 | train loss 10.2163 | val loss 10.2213
step  200 | train loss 7.6244 | val loss 8.0573
step  400 | train loss 7.2353 | val loss 7.8476
step  600 | train loss 6.8257 | val loss 7.5981
step  800 | train loss 6.5372 | val loss 7.5352
step 1000 | train loss 6.2887 | val loss 7.4649
step 1200 | train loss 6.1024 | val loss 7.4844
step 1400 | train loss 5.8433 | val loss 7.5154
step 1600 | train loss 5.6793 | val loss 7.5420
step 1800 | train loss 5.4391 | val loss 7.6265
step 2000 | train loss 5.1677 | val loss 7.5609
step 2200 | train loss 4.9925 | val loss 7.7044
step 2400 | train loss 4.7195 | val loss 7.7696
step 2600 | train loss 4.4311 | val loss 7.8101
step 2800 | train loss 4.2647 | val loss 7.9222


In [ ]:
# =========================
# 8. GENERATE UNTIL <eos>
# =========================
@torch.no_grad()
def generate(model, start_str, max_new_tokens=50, temperature=1.0, top_k=None):
    model.eval()

    start_ids = encode_text_to_ids(start_str)
    if len(start_ids) == 0:
        start_ids = [UNK_ID]

    context = torch.tensor(start_ids, dtype=torch.long, device=device).unsqueeze(0)

    i = 0
    for _ in range(max_new_tokens):
        idx_cond = context[:, -model.block_size:]   # cắt bớt nếu quá dài
        logits, _ = model(idx_cond)

        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        # print(probs)

        context = torch.cat([context, next_token], dim=1)

        # if next_token.item() == EOS_ID:
        #     break

    return decode(context[0].tolist())


In [ ]:
# =========================
# 9. TEST
# =========================
print("\n=== GENERATED TEXT ===\n")
print(generate(model, "ROMEO:", max_new_tokens=40, temperature=0.9, top_k=20))
print("\n----------------------\n")
print(generate(model, "JULIET:", max_new_tokens=40, temperature=0.9, top_k=20))


=== GENERATED TEXT ===

ROMEO: I think I was the law: I never did recoil and thirty, a bloody guest That would be a happy thing: the king's sister would be the sea to the public weal: obey, thou shalt be gone. QUEEN ELIZABETH: Harp

----------------------

JULIET: Nurse: But I speak the matter:--Nurse, I would tell thee, tell thee not the sin I'll take a happy man. DUKE VINCENTIO: For what to me what I speak no pheasant, cock nor hen. ISABELLA: To he be to be
